# Эмбеддинги молекул на GPU (Colab)

Считаем эмбеддинги предобученных химических моделей для всех молекул проекта и скачиваем один архив.
Данные соревнования в Colab **не загружаются**: нужны два небольших файла — список стандартизованных SMILES (≈ 2 600 строк) и сам скрипт.
Репозиторий клонировать не нужно: скрипт не импортирует код проекта, а в GitHub может лежать устаревшая версия.

**До Colab (у себя):** `python scripts/export_smiles.py` → файл `colab/smiles_std.csv`.
**В ячейке 3 загрузите оба файла сразу:** `colab/smiles_std.csv` и `scripts/colab_embeddings.py`.
**После Colab (у себя):** распаковать архив в `data/embeddings/`, затем
`python scripts/make_embedding_configs.py`, `python -m vacv_si cv -c configs/exp/emb_final.yaml` и
`python -m vacv_si ensemble -c configs/exp/emb_final.yaml --no-submit` (см. README и `experiments/README.md`, № 26).

Среда: `Runtime → Change runtime type → GPU`.

In [ ]:
import torch, subprocess
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
# 2) зависимости (torch в Colab уже есть). Если Colab попросит перезапустить среду - перезапустите и начните с этой ячейки ещё раз.
!pip install -q transformers "chemprop>=2.2.0" rdkit pandas numpy tqdm selfies

In [ ]:
# 3) загрузите ДВА файла сразу: smiles_std.csv и colab_embeddings.py (кнопка выбора файлов появится под ячейкой)
from google.colab import files
import os, shutil
os.makedirs("colab", exist_ok=True); os.makedirs("scripts", exist_ok=True)
for name in files.upload():
    shutil.move(name, "colab/smiles_std.csv" if name.startswith("smiles_std") else "scripts/colab_embeddings.py")
import pandas as pd
print(len(pd.read_csv("colab/smiles_std.csv")), "molecules |", os.path.exists("scripts/colab_embeddings.py") and "script ok")

In [ ]:
# 4) эмбеддинги. ChemBERTa-77M (MLM и MTR) и CheMeleon работают без чужого кода. MoLFormer (--trust-remote-code) ВЫПОЛНЯЕТ код,
#    лежащий в его репозитории на HuggingFace: в Colab это изолированная виртуальная машина, поэтому здесь это приемлемо.
#    Занимает несколько минут. Если в выводе есть "[molformer] FAILED" - см. следующую ячейку.
!python scripts/colab_embeddings.py --models chemberta77m_mlm chemberta77m_mtr chemeleon molformer chemberta_zinc gpt2_zinc87m --trust-remote-code --device cuda --batch 256

In [ ]:
# 5) если MoLFormer упал из-за версии transformers, запустите его на проверенной версии (раскомментируйте и выполните;
#    потом, при необходимости, перезапустите среду):
# !pip install -q "transformers==4.44.2"
# !python scripts/colab_embeddings.py --models molformer --trust-remote-code --device cuda --batch 256
!cat data/embeddings/manifest.json

In [ ]:
# 6) необязательно и долго: ChemGPT-1.2B (скачивается ~5 ГБ; читает SELFIES, скрипт конвертирует SMILES сам). Ожидаемый прирост мал.
# !python scripts/colab_embeddings.py --models chemgpt1p2b --device cuda --batch 64

In [ ]:
# 7) скачать результат одним архивом -> распаковать в data/embeddings/ у себя
!cd data/embeddings && zip -q ../../embeddings.zip *.npz manifest.json
files.download("embeddings.zip")